In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/ch08-umap-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["deepchem==2.8.1.dev20260821122258", "numpy==1.26.4", "scipy==1.17.1",
            "rdkit==2026.9.1", "scikit-learn==1.9.1", "pandas==3.0.6", "matplotlib==3.11.2",
            "umap-learn==0.5.12", "llvmlite==0.50.0", "numba==0.68.1",
            "pynndescent==0.7.0", "tqdm==4.70.1"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "08-03-tsne-and-umap.py"

從 ESOL 保留兩個芳香族分子，對照 UMAP 圖面最近鄰與 Morgan 指紋最近鄰。首次執行需連網下載資料與安裝套件；Numba 首次編譯會多花一些時間。小練習可加在下一格 Python 腳本末尾。

In [ ]:
%%writefile "{script}"
from collections import Counter
from pathlib import Path
import os
import tempfile

import deepchem as dc
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from rdkit import Chem, DataStructs
from rdkit.Chem import Draw, rdFingerprintGenerator
from sklearn.manifold import TSNE
from umap import UMAP

seed = 42
cache = Path(os.environ.get("DEEPCHEM_DATA_DIR", Path(tempfile.gettempdir()) / "cheminfo-datasets"))
cache.mkdir(parents=True, exist_ok=True)
# ESOL 的 1,128 筆實測 log10(S / mol L⁻¹)；Delaney, DOI 10.1021/ci034243x。
# 此處比對指定分子的鄰居，不評估預測分數，故手動保留兩筆而不另切測試集。
tasks, splits, _ = dc.molnet.load_delaney(
    featurizer=dc.feat.RawFeaturizer(smiles=True), splitter=None, transformers=[],
    data_dir=str(cache), reload=False)
data, = splits
assert len(data) == 1128 and tasks == ["measured log solubility in mols per litre"]

# 只收單一片段、有芳香原子的分子。相同 canonical SMILES 若有多筆，整組略過。
rows = []
for raw, y, w in zip(data.ids, data.y[:, 0], data.w[:, 0]):
    mol = Chem.MolFromSmiles(str(raw).strip())
    if mol is None or len(Chem.GetMolFrags(mol)) != 1 or not any(a.GetIsAromatic() for a in mol.GetAtoms()):
        continue
    if np.isfinite(y) and w > 0:
        rows.append((Chem.MolToSmiles(mol), float(y)))
counts = Counter(s for s, y in rows)
values = {s: y for s, y in rows if counts[s] == 1}
held = ["COc1ccccc1", "CCCc1ccccc1"]  # 苯甲醚、正丙苯；比對前不使用其 log S
assert all(s in values for s in held)
known = sorted(values.keys() - set(held))
gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
fps = [gen.GetFingerprint(Chem.MolFromSmiles(s)) for s in known + held]
X = np.asarray([np.asarray(fp, dtype=bool) for fp in fps])
X_known, X_held = X[:len(known)], X[len(known):]

# t-SNE 只看已知組。UMAP 先固定已知組的圖，再放上候選分子。
tsne = TSNE(metric="jaccard", init="random", perplexity=30, random_state=seed)
zt = tsne.fit_transform(X_known)
mapper = UMAP(metric="jaccard", n_neighbors=15, min_dist=0.15,
              random_state=seed, transform_seed=seed, n_jobs=1)
zu = mapper.fit_transform(X_known)
zh = mapper.transform(X_held)
assert np.isfinite(zt).all() and np.isfinite(zu).all() and np.isfinite(zh).all()

# 所有鄰居都從已知組取。距離同分時依已排序的 SMILES 決定順序。
def neighbors(i):
    similarities = np.array(DataStructs.BulkTanimotoSimilarity(fps[len(known) + i], fps[:len(known)]))
    plane_order = np.argsort(np.linalg.norm(zu - zh[i], axis=1), kind="stable")
    bit_order = np.argsort(-similarities, kind="stable")
    return plane_order, bit_order, similarities

print(f"known={len(known)}, held={len(held)}")
plane, bits, sim = neighbors(0)
for label, j in [("UMAP", plane[0]), ("Morgan", bits[0])]:
    print(label, known[j], f"Tanimoto={sim[j]:.3f}", f"log S={values[known[j]]:.2f}")
print("Anisole log S (revealed after lookup):", values[held[0]])

fig, axes = plt.subplots(1, 2, figsize=(9.2, 4.1), layout="constrained")
for ax, z, name in zip(axes, (zt, zu), ("t-SNE", "UMAP")):
    dots = ax.scatter(*z.T, s=13, c=[values[s] for s in known], cmap="viridis", alpha=0.87, linewidths=0)
    ax.set_title(name)
    ax.set(xlabel=f"{name} 1", ylabel=f"{name} 2", aspect="equal")
    ax.set_xticks([])
    ax.set_yticks([])
for i, (name, marker) in enumerate([("Anisole", "*"), ("Propylbenzene", "D")]):
    axes[1].scatter(*zh[i], marker=marker, s=80, color="#ef813a", edgecolors="black", linewidths=0.9, label=name)
axes[1].legend(loc="best", fontsize=8)
fig.colorbar(dots, ax=axes, label="Measured log S (S in mol/L)", shrink=0.75)
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
figure_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(figure_dir / "08-03-tsne-and-umap.png", dpi=170)
plt.close(fig)

structures = [held[0], known[plane[0]], known[bits[0]]]
labels = ["Query: anisole", f"UMAP nearest  |  T = {sim[plane[0]]:.3f}",
          f"Morgan nearest  |  T = {sim[bits[0]]:.3f}"]
panel = Draw.MolsToGridImage([Chem.MolFromSmiles(s) for s in structures], legends=labels,
                             molsPerRow=3, subImgSize=(325, 215))
panel.save(figure_dir / "08-03-neighbors.png")


In [ ]:
from IPython.display import Image, display

subprocess.run([python, str(script)], check=True)
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
display(Image(filename=str(figure_dir / "08-03-tsne-and-umap.png")))
display(Image(filename=str(figure_dir / "08-03-neighbors.png")))